# 04 · ลายเซ็นก๊าซและจุดความร้อน (หลักฐานชุด "เพราะอะไร")

- เผาชีวมวล: PM2.5 สูง พร้อม TVOC สูง
- การจราจร: PM2.5 สูง พร้อมดัชนี CO (MQ-7) สูง
- จัดกลุ่มเหตุการณ์ PM2.5 สูงด้วย KMeans แล้วตรวจกับจำนวนจุดความร้อน NASA FIRMS ต้นลม

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# หา root ของ repo (โฟลเดอร์ที่มี data/) ไม่ว่าจะเปิด notebook จากที่ไหน
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

dev = pd.read_csv(PROC / "device_readings.csv", parse_dates=["time"], index_col="time")
GAS = [c for c in ["tvoc", "co"] if c in dev and dev[c].notna().any()]
if not GAS:
    print("ยังไม่มีข้อมูลก๊าซจากเครื่อง (ENS160 / MQ-7) ข้ามการจัดกลุ่ม ไปดูส่วนจุดความร้อนด้านล่างได้")
else:
    h = dev[["pm25_corr"] + GAS].resample("h").median().dropna()
    events = h[h["pm25_corr"] > h["pm25_corr"].quantile(0.75)].copy()     # ชั่วโมงที่ฝุ่นสูง
    Z = StandardScaler().fit_transform(np.log1p(events))
    events["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
    display(events.groupby("cluster").median().round(1))

## จุดความร้อนต้นลม

สร้างไฟล์ด้วย `python scripts/fetch_firms.py --start ... --end ...`
ทิศลมของ Open-Meteo คือทิศที่ลม **พัดมา** จุดความร้อนที่อยู่ในทิศนั้น ±45° ภายใน 300 กม. นับเป็น "ต้นลม" 

In [ ]:
KK = (16.4419, 102.8360)

def bearing(lat1, lon1, lat2, lon2):
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    return (np.degrees(np.arctan2(np.sin(dl) * np.cos(p2),
            np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl))) + 360) % 360

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

firms_files = sorted(RAW.glob("firms_*.csv"))
if firms_files:
    fire = pd.concat(pd.read_csv(f) for f in firms_files).drop_duplicates()
    fire["date"] = pd.to_datetime(fire["acq_date"])
    fire["bearing"] = bearing(*KK, fire["latitude"], fire["longitude"])
    fire["dist_km"] = haversine_km(*KK, fire["latitude"], fire["longitude"])
    met = pd.read_csv(PROC / "hourly_khonkaen.csv", parse_dates=["time"], index_col="time")
    wd = met["wind_direction_10m"].resample("D").apply(
        lambda s: (np.degrees(np.arctan2(np.sin(np.radians(s)).mean(), np.cos(np.radians(s)).mean())) + 360) % 360)
    fire["wind_from"] = wd.reindex(fire["date"]).to_numpy()
    diff = (fire["bearing"] - fire["wind_from"] + 180) % 360 - 180
    fire["upwind"] = (diff.abs() <= 45) & (fire["dist_km"] <= 300)
    upwind_daily = fire[fire["upwind"]].groupby("date").size().rename("upwind_hotspots")
    daily_pm = met["station_pm25"].resample("D").mean()
    comp = pd.concat([daily_pm, upwind_daily], axis=1).fillna({"upwind_hotspots": 0})
    print("Spearman r =", round(comp.corr(method="spearman").iloc[0, 1], 2))
    comp.plot(secondary_y="upwind_hotspots", figsize=(10, 4)); plt.show()